In [1]:
def to_cosmographic(H0, Om, Ok, w):
    q0 = 1.0/2.0 *(1.0-Ok-3.0*(-1.0+Om+Ok)*w)
    j0 = 1.0/2.0 * (2.0 * Om + (1.0 - Om - Ok)*(2+9*w*(1+w)))
    s0 = 1.0/4.0 * (2*(7-Ok)*(Ok-1.0)+3*(5*Ok-27)*(1-Om-Ok)*w+9*(1-Om-Ok)*(-16+4*Ok+3*Om)*w**2 + 27.0*(-3.0+Ok+Om)*(1-Ok-Om)*w**3)
    return H0, q0, j0, s0

In [2]:
def z_series(z, H0, q0, j0, s0):

    a0 = 1.0
    a1 = 1.0/2.0 - q0/2.0
    a2 = -1.0/6.0 - j0/6.0 + q0/6.0 + q0**2.0 / 2.0
    a3 = 1.0/12.0 + 5 * j0 / 24.0 - q0/12.0 + 5.0*j0*q0/12.0  -5.0 * q0**2.0 / 8.0  -5.0 * q0**3.0 / 8.0 + s0/24.0

    return 300000 * z / H0 * (a0 + a1*z + a2 * z**2.0 + a3 * z**3.0)

def chi_square_z(theta, z, data):
    H0, q0, j0, s0 = theta
    diff = z_series(z, H0, q0, j0, s0) - data
    return np.sum(diff*diff)

In [3]:
def y_series(y, H0, q0, j0, s0):

    a0 = 1.0
    a1 = -1/2*(q0-3)
    a2 = 1/6 * (12-5*q0+3*q0**2-j0)
    a3 = 1/24 * (50 - 7*j0 -26*q0+10*q0*j0+21*q0**2-15*q0**3+s0)

    return 300000 * y / H0 * (a0 + a1*y + a2 * y**2.0 + a3 * y**3.0)

def chi_square_y(theta, y, data):
    H0, q0, j0, s0  = theta
    diff = y_series(y, H0, q0, j0, s0) - data
    return np.sum(diff*diff)

In [4]:
def log_series(log1z, H0, q0, j0, s0):

    a0p = 1.0
    a1p = 1 -1/2*q0
    a2p = 1/6 * (3-2*q0+3*q0**2-j0)
    a3p =  1/6 +(1/8)*(-q0 + q0**2 -5*q0**3) +(5/12)*q0*j0 + 1/24*(-j0 + s0)

    return 300000 * log1z / H0 * (a0p + a1p*log1z + a2p * log1z**2.0 + a3p * log1z**3.0)

def chi_square_log(theta, log1z, data):
    H0, q0, j0, s0  = theta
    diff = log_series(log1z, H0, q0, j0, s0) - data
    return np.sum(diff*diff)

In [5]:
def dLpade22(z,H0, q0, j0, s0):
    numerator = (6 *z* (10 + 9 * z - 6 * (q0**3) * z  
                        +s0 * z - 2 * (q0**2) * (3 + 7 * z) - q0 * (16 + 19 * z) + j0 * (4 + (9 + 6 * q0) * z)))
    denominator = (60 + 24 * z + 6 * s0 * z - 2 * z**2+ 4 * (j0**2) * z**2 - 9 * (q0**4) * z**2 - 3 * s0 * z**2 
                   + 6 * q0**3 * z * (-9 + 4 * z)  + q0**2 * (-36 - 114 * z + 19 * z**2) + j0 * (24 + 6 * (7 + 8 * q0) * z 
                   + (-7 - 23 * q0 + 6 * q0**2) * z**2) + q0 * (-96 - 36 * z + (4 + 3 * s0) * z**2))

    Pz_value =  (300000/H0)*(numerator / (denominator ) )
    return Pz_value

def lkl_pade22(pars,z,data):
    y=data
    x=dLpade22(z,*pars)
    return np.sum((y-x)**2)

In [6]:
def dLpade21(z,H0, q0, j0):
    numerator = z * (6 * (-1 + q0) + (-5 -2*j0 +q0 * (8 + 3*q0))*z)
    denominator = -2 * (3 + z + j0*z) + 2*q0 * (3 + z + 3*q0*z)

    P2z_value =  (300000/H0)*(numerator / (denominator ) )
    return P2z_value

def lkl_pade21(pars,z,data):
    y=data
    x=dLpade21(z,*pars)
    return np.sum((y-x)**2)